<h1><center>Laboratorio 6: Optimización de modelos 🧪</center></h1>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos - Primavera 2025</strong></center>

### Cuerpo Docente:

- Profesores: Diego Cortez, Gabriel Iturra
- Auxiliares: Melanie Peña, Valentina Rojas
- Ayudantes: Nicolás Cabello, Cristopher Urbina

### Equipo: poplolitas SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Laura Maldonado
- Nombre de alumno 2: Javiera Arévalo


Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda **fuertemente** asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

### **Link de repositorio de GitHub:** [Insertar Repositorio](https://github.com/lauraflm/MDS7202-Laboratorio-de-Programacion-Cientifica-para-Ciencia-de-Datos)

### Temas a tratar

- Predicción de demanda usando `xgboost`
- Búsqueda del modelo óptimo de clasificación usando `optuna`
- Uso de pipelines.


### Reglas:

- **Grupos de 2 personas**
- Fecha de entrega: 6 días de plazo con descuento de 1 punto por día. Entregas Martes a las 23:59.
- Instrucciones del lab el viernes a las 16:15 en formato online. Asistencia no es obligatoria, pero se recomienda fuertemente asistir.
- <u>Prohibidas las copias</u>. Cualquier intento de copia será debidamente penalizado con el reglamento de la escuela.
- Tienen que subir el laboratorio a u-cursos y a su repositorio de github. Labs que no estén en u-cursos no serán revisados. Recuerden que el repositorio también tiene nota.
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Pueden usar cualquier material del curso que estimen conveniente.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias útiles

In [ ]:
!pip install -qq xgboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 6.8 MB/s eta 0:00:00


# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementación de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corpóreo **Fiu** se anima y decide levantar su propio negocio de consultoría en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset señalado y visualice a través de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempeño en el proyecto de caracterización de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv("sales.csv")

print(df.head())
print(df.shape)

   id      date    city       lat      long     pop    shop        brand  \
0   0  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
1   1  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
2   2  31/01/12  Athens  37.97945  23.71622  672130  shop_1  kinder-cola   
3   3  31/01/12  Athens  37.97945  23.71622  672130  shop_1   adult-cola   
4   4  31/01/12  Athens  37.97945  23.71622  672130  shop_1   adult-cola   

  container capacity  price  quantity  
0     glass    500ml   0.96     13280  
1   plastic    1.5lt   2.86      6727  
2       can    330ml   0.87      9848  
3     glass    500ml   1.00     20050  
4       can    330ml   0.39     25696  
(7456, 12)


## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su magíster en ciencia de datos y recuerda que debe seguir una serie de *buenas prácticas* para entrenar correcta y debidamente su modelo. Después de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el día, mes y año de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos numéricos y categóricos. Use `OneHotEncoder` para las variables categóricas. `Nota:` Utilice el método `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como último paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la métrica `mean_absolute_error` sobre los datos de validación. ¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los parámetros por default**. ¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [ ]:
from sklearn import set_config
from sklearn.model_selection import train_test_split
set_config(transform_output="pandas")


RANDOM_STATE = 22

X = df.drop(columns=['quantity'])
y = df['quantity']


X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, shuffle=True)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=1/3, random_state=RANDOM_STATE, shuffle=True)

print({
    "train": X_train.shape[0],
    "val": X_val.shape[0],
    "test": X_test.shape[0]
})


{'train': 5219, 'val': 1491, 'test': 746}


In [ ]:
#2.Implemente un FunctionTransformer para extraer el día, mes y año de la variable date.
#Guarde estas variables en el formato categorical de pandas. [1 punto]

from sklearn.preprocessing import FunctionTransformer

# Función para extraer el día, mes y año
def extract_date(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['day'] = df['date'].dt.day.astype('category')
    df['month'] = df['date'].dt.month.astype('category')
    df['year'] = df['date'].dt.year.astype('category')
    return df

# Crear el FunctionTransformer
date_transformer = FunctionTransformer(
    extract_date,
    validate=False
)
date_transformer.set_output(transform="pandas")


FunctionTransformer(func=<function extract_date at 0x7aca0877ec00>)

In [ ]:
#3.Implemente un ColumnTransformer para procesar de manera adecuada los datos numéricos y categóricos.
#Use OneHotEncoder para las variables categóricas. Nota: Utilice el método .set_output(transform='pandas')
# para obtener un DataFrame como salida del ColumnTransformer [1 punto]

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Columnas numéricas y categóricas
numerical_features = ['lat', 'long', 'pop', 'price']
categorical_features = ['city', 'shop', 'brand', 'container', 'capacity','date']

# Crear el ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(sparse_output=False), categorical_features)],
          sparse_threshold=0,).set_output(transform='pandas')

In [ ]:
#4.Guarde los pasos anteriores en un Pipeline, dejando como último paso el regresor DummyRegressor
#para generar predicciones en base a promedios. [0.5 punto]
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor

features = ['lat','long','pop','price','city','shop','brand','container','capacity','date']
# Crear el pipeline con el preprocessor del paso anterior
dummy_pipeline = Pipeline([
    ("date_parts", date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])

# Entrenar el pipeline
dummy_pipeline.fit(X_train, y_train)

# Predecir sobre el conjunto de validación
preds_val = dummy_pipeline.predict(X_val)

# Mostrar algunas predicciones
print(preds_val[:10])


[29403.79517149 29403.79517149 29403.79517149 29403.79517149
 29403.79517149 29403.79517149 29403.79517149 29403.79517149
 29403.79517149 29403.79517149]


/tmp/ipython-input-3837809157.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])
/tmp/ipython-input-3837809157.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


In [ ]:
#5. Entrene el pipeline anterior y reporte la métrica mean_absolute_error sobre los datos de validación.
#¿Cómo se interpreta esta métrica para el contexto del negocio? [0.5 puntos]

from sklearn.metrics import mean_absolute_error

# Realizar predicciones sobre el conjunto de validación
val_predictions = dummy_pipeline.predict(X_val)

# Calcular la métrica Mean Absolute Error (MAE)
dummy_mae = mean_absolute_error(y_val, val_predictions)

# Mostrar el resultado
print(f"Mean Absolute Error (MAE) sobre datos de validación: {dummy_mae:.2f}")


Mean Absolute Error (MAE) sobre datos de validación: 13503.88


/tmp/ipython-input-3837809157.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


In [ ]:
#6.Finalmente, vuelva a entrenar el Pipeline pero esta vez usando XGBRegressor como modelo utilizando los parámetros por default.
#¿Cómo cambia el MAE al implementar este algoritmo? ¿Es mejor o peor que el DummyRegressor? [1 punto]
from xgboost import XGBRegressor
# XGBRegressor
xgb_pipeline = Pipeline([
    ("date_parts", date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=RANDOM_STATE))
])

# Ajustar el modelo XGBRegressor
xgb_pipeline.fit(X_train[features],y_train)

# Predicciones y cálculo del MAE
xgb_predictions = xgb_pipeline.predict(X_val)
xgb_mae = mean_absolute_error(y_val, xgb_predictions)
print(f"MAE del XGBRegressor: {xgb_mae:.2f}")

/tmp/ipython-input-3837809157.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


MAE del XGBRegressor: 3642.49


/tmp/ipython-input-3837809157.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


In [ ]:
#7.
import joblib
joblib.dump(dummy_pipeline, 'dummy_regressor_model.pkl')
joblib.dump(xgb_pipeline, 'xgb_regressor_model.pkl')

['xgb_regressor_model.pkl']

El MAE representa el error absoluto promedio entre las predicciones y los valores reales, expresado en las mismas unidades que la variable objetivo (quantity). En este caso, indica cuánto se equivoca el modelo, en promedio, al estimar la cantidad real de unidades vendidas.

El DummyRegressor actúa como modelo base (baseline), ya que su predicción consiste únicamente en el promedio global de la variable objetivo. Por ello, su desempeño es limitado y sirve principalmente para establecer un punto de referencia inicial.

Por el contrario, el XGBRegressor utiliza un enfoque basado en árboles de decisión y boosting, lo que le permite capturar relaciones no lineales entre las variables predictoras (price, city, shop, brand, entre otras). Gracias a esto, logra reducir el error medio a cerca de la mitad del obtenido con el modelo base (se pasa de 13546 a 3642).

El XGBRegressor muestra un desempeño considerablemente superior al DummyRegressor. Por lo tanto, desde una perspectiva de negocio, esta diferencia es importante, ya que hay que tener en consideración que un menor MAE implica predicciones más precisas, lo que se traduce en una mejor estimación de la demanda o de las ventas esperadas.

## 2. Forzando relaciones entre parámetros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la economía le *sopla* que la demanda guarda una relación inversa con el precio del producto. Motivado para impresionar al querido corpóreo, se propone hacer uso de esta información para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relación monótona negativa entre el precio y la cantidad. Para aplicar esta restricción apóyese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentación</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser así, probablemente le sea útil **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el método `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validación. [1 puntos]

3. ¿Cómo cambia el error al incluir esta relación? ¿Tenía razón su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [ ]:
#6. Vuelva a entrenar el Pipeline con XGBRegressor, forzando una relación monótona negativa entre el precio y la cantidad.
#Para aplicar esta restricción, utilice la documentación de XGBoost y el nombre de las variables del preprocesamiento. [6 puntos]

from xgboost import XGBRegressor
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import joblib


# Obtener los nombres de las columnas transformadas
preprocessor.set_output(transform='pandas')
X_train_transformed = preprocessor.fit_transform(X_train[features])
feature_names = X_train_transformed.columns

# índice de la variable 'price' en las columnas transformadas
monotone_constraints = [0] * len(feature_names)
price_index = np.where(feature_names == 'num__price')[0][0]
monotone_constraints[price_index] = -1  # relación negativa entre precio y cantidad

#  modelo XGBRegressor + restricción
xgb_model = XGBRegressor(
    random_state=42,
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    monotone_constraints=tuple(monotone_constraints)
)

# pipeline monotono
pipeline_monotone = Pipeline([
    ('date_parts', date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', xgb_model)
])


pipeline_monotone.fit(X_train, y_train)
val_predictions_monotone = pipeline_monotone.predict(X_val)

# mae monotono
mae_monotone = mean_absolute_error(y_val, val_predictions_monotone)

print(f"Mean Absolute Error (MAE) con restricción monótona: {mae_monotone:.2f}")

# Mpdelo en un archivo .pkl
joblib.dump(pipeline_monotone, 'xgb_regressor_monotonic.pkl')


/tmp/ipython-input-3837809157.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


Mean Absolute Error (MAE) con restricción monótona: 3562.33


/tmp/ipython-input-3837809157.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


['xgb_regressor_monotonic.pkl']

Después de imponer una relación monótona negativa entre el precio y la cantidad, el modelo obtuvo un MAE de 3562, mejorando respecto al modelo anterior sin restricción, que había alcanzado un MAE de 3642.

Esta reducción del error confirma que la relación inversa entre ambas variables efectivamente está presente en los datos, validando la intuición económica de que a mayor precio, menor demanda esperada. Por lo que, el forzar esta relación es beneficioso.

Además, es importante mencionar que incluir la restricción introduce un componente de conocimiento "experto" en el modelo lo que ayuda a la interpretabilidad y la robustez del modelo, al evitar predicciones contraintuitivas como aumentos simultáneos de precio y demanda.


## 1.3 Optimización de Hiperparámetros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun más* su modelo. En particular, le comenta de la optimización de hiperparámetros con metodologías bayesianas a través del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuración obtenida en la sección anterior, utilice `optuna` para optimizar sus hiperparámetros. En particular, se pide que su optimización considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como método de muestreo
- De `XGBRegressor`, optimice los siguientes hiperparámetros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperparámetro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una función `objective()` que permita minimizar el `MAE` en el conjunto de validación. Use el método `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
4. Explique cada hiperparámetro y su rol en el modelo. ¿Hacen sentido los rangos de optimización indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [ ]:
#7. Optimización bayesiana con Optuna (TPESampler) reutilizando el pipeline guardado

import joblib
import optuna
from optuna.samplers import TPESampler
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

set_config(transform_output="pandas")

base_pipe = joblib.load("xgb_regressor_monotonic.pkl")
base_pre  = base_pipe.named_steps["preprocessor"]  # ColumnTransformer

# Recuperar listas de columnas desde el preprocessor guardado
numerical_features = []
categorical_features = []
for name, trans, cols in base_pre.transformers_:
    if name == "num":
        numerical_features = list(cols)
    elif name == "cat":
        categorical_features = list(cols)

target = "quantity"
features = numerical_features + categorical_features


# armar preprocessor por trial
def armar_prepro_trial(min_freq: float) -> ColumnTransformer:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=min_freq)
    pre = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", ohe, categorical_features),
        ]
    )
    pre.set_output(transform="pandas")
    return pre

# objective(): minimiza MAE en validación, respeta monotone constraint en 'price' y guarda el pipeline
def objective(trial: optuna.Trial) -> float:
    learning_rate    = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators     = trial.suggest_int("n_estimators", 50, 1000)
    max_depth        = trial.suggest_int("max_depth", 3, 10)
    max_leaves       = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha        = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda       = trial.suggest_float("reg_lambda", 0.0, 1.0)
    ohe_min_freq     = trial.suggest_float("min_frequency", 0.0, 1.0)  # OneHotEncoder

    # Preprocesador del trial
    pre_t = armar_prepro_trial(min_freq=ohe_min_freq)

    # Obtener nombres transformados para alinear constraint monótono
    Xtr = pre_t.fit_transform(X_train)
    feat_names = list(Xtr.columns)

    # Vector de constraints: -1 para 'price' (relación inversa), 0 resto
    mono = [0] * len(feat_names)
    for i, name in enumerate(feat_names):
        if name == "num__price" or name.endswith("price"):
            mono[i] = -1

    # Modelo XGB con restricción monótona
    xgb = XGBRegressor(
        random_state=RANDOM_STATE,
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        monotone_constraints=tuple(mono),
        tree_method="hist",
        verbosity=0
    )

    # Pipeline del trial
    pipe = Pipeline([
        ("date_parts", date_transformer),
        ("preprocessor", pre_t),
        ("regressor", xgb),
    ])

    pipe.fit(X_train, y_train)

    # MAE en validación
    pred_val = pipe.predict(X_val)
    mae = mean_absolute_error(y_val, pred_val)

    # Guardar pipeline entrenado en user_attr
    trial.set_user_attr("pipeline", pipe)

    return mae

#  timeout 5 minutos
sampler = TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, timeout=300, n_jobs=1)  # 300 s = 5 min

#  guardado .pkl
best_trial = study.best_trial
print(f"Trials realizados: {len(study.trials)}")
print(f"Mejor MAE (validación): {best_trial.value:.4f}")
print("Mejores hiperparámetros:")
for k, v in best_trial.params.items():
    print(f"  - {k}: {v}")

best_pipeline = best_trial.user_attrs["pipeline"]
joblib.dump(best_pipeline, "xgb_monotone_optuna.pkl")
print("Modelo guardado en: xgb_monotone_optuna.pkl")


[I 2025-10-08 13:48:11,961] A new study created in memory with name: no-name-a2ce4706-f1e2-4b18-a9e2-9bc158802a66
/tmp/ipython-input-3837809157.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])
/tmp/ipython-input-3837809157.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])
[I 2025-10-08 13:48:13,612] Trial 0 finished with value: 8525.7880859375 and parameters: {'learning_rate': 0.021637593198525422, 'n_estimators': 508, 'max_depth': 6, 'max_leaves': 86, 'min_child_weight': 1, 'reg_alpha': 0.3388639606037004, 'reg_lambda': 0.2705328332978312, 'min_frequency': 0.6910413504495961}. Best is trial 0 with value: 8525.7880859375.
/tmp/

Trials realizados: 244
Mejor MAE (validación): 3301.5510
Mejores hiperparámetros:
  - learning_rate: 0.08965154370928277
  - n_estimators: 471
  - max_depth: 10
  - max_leaves: 82
  - min_child_weight: 3
  - reg_alpha: 0.4903874595178815
  - reg_lambda: 0.5223753638936213
  - min_frequency: 0.0008395548387055019
Modelo guardado en: xgb_monotone_optuna.pkl


Tras ejecutar la optimización bayesiana con Optuna sobre el modelo XGBRegressor con restricción monótona negativa en la variable de precio (price), se alcanzó un MAE de 3301.55 en el conjunto de validación.
Mejores hiperparámetros encontrados:
learning_rate: 0.08965
n_estimators: 471
max_depth: 10
max_leaves: 82
min_child_weight: 3
reg_alpha: 0.49039
reg_lambda: 0.52238
min_frequency (OneHotEncoder): 0.00084

learning_rate = 0.08965. Es un paso de aprendizaje moderado permite entrenar de forma estable sin grandes saltos.

n_estimators = 471. Cantidad medio-alta de árboles, que hace sentido con el learning_rate, ya que se toman muchos pasos pequeños que van mejorando la predicción.

max_depth = 10. Esto es una profundidad alta que entrega mucha capacidad para capturar no linealidades e interacciones.

max_leaves = 82. Limita el número de hojas por árbol (es un alto número).

min_child_weight = 3. Umbral moderado que evita splits con poca evidencia. Es un buen contrapeso para la profundidad 10 porque frena divisiones débiles y ayuda a generalizar.

reg_alpha = 0.49039 (L1). Regularización L1 media-alta que favorece “sparsity” en ramas/pesos.

reg_lambda = 0.52238 (L2). Regularización L2 media que suaviza los pesos y aporta estabilidad adicional.

min_frequency (OneHotEncoder) = 0.00084. Este es un umbral muy bajo, por lo que se preservan todas las categorías raras. Si el rendimiento en test cae respecto a validación, consideraría subirlo un poco (p. ej., 0.005–0.02) para agrupar infrecuentes.

Por lo tanto, es un modelo con tasa de aprendizaje moderada y muchos árboles, lo que permite un aprendizaje progresivo. La profundidad alta (10) y el número elevado de hojas (max_leaves = 82) entregan gran capacidad para capturar interacciones no lineales; esto se equilibra con regularización L1/L2 y un min_child_weight que evita divisiones poco informativas. El min_frequency muy bajo conserva categorías raras (más señal potencial, a costa de mayor dimensionalidad). La restricción monótona en price asegura coherencia de negocio: al subir el precio, la cantidad predicha no aumenta.

El ajuste bayesiano mejoró el desempeño frente a las versiones base y mantuvo la consistencia impuesta por la monotonía en price.

Comparando con las etapas previas, el modelo ha mostrado una mejora progresiva del desempeño:
DummyRegressor: MAE = 13503
XGBRegressor: MAE = 3642
XGB con restricción monótona: MAE = 3562
XGB optimizado con Optuna: MAE = 3301


## 4. Optimización de Hiperparámetros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Después de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en sí mismo. Después de leer un par de post de personas de dudosa reputación en la *deepweb*, usted llega a la conclusión que puede cumplir este objetivo mediante la implementación de **Prunning**.

Vuelva a optimizar los mismos hiperparámetros que la sección pasada, pero esta vez utilizando **Prunning** en la optimización. En particular, usted debe:

- Responder: ¿Qué es prunning? ¿De qué forma debería impactar en el entrenamiento? [2 puntos]
- Redefinir la función `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como método de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el número de *trials*, el `MAE` y los mejores hiperparámetros encontrados. ¿Cómo cambian sus resultados con respecto a la sección anterior? ¿A qué se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opción anterior, pueden especificar `show_progress_bar = True` en el método `optimize` para *más sabor*.

Hint: Si quieren especificar parámetros del método .fit() del modelo a través del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementación

In [ ]:
!pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 2.7 MB/s eta 0:00:00


¿Qué es pruning? ¿Cómo impacta en el entrenamiento?

"Pruning" (poda) en Optuna es una técnica de parada temprana a nivel de trial:
durante la optimización, cada trial se entrena de forma incremental y se monitoriza su métrica intermedia (p. ej., MAE en el set de validación a lo largo de los árboles).

Si un trial muestra rendimiento claramente peor que la mediana de los mejores en ese
mismo punto (u otro criterio del pruner), Optuna lo "poda" (detiene) antes de que consuma todo el presupuesto de cómputo.

Impacto esperado:
- Reduce tiempo total de búsqueda (termina rápido candidates malos).
- Permite explorar más configuraciones en el mismo tiempo (más trials efectivos).
- Mantiene o mejora calidad final: aunque se podan muchos trials, aquellos con métricas prometedoras siguen entrenando hasta completar y potencian encontrar mejores hiperparámetros. En resumen: más eficiencia sin sacrificar rendimiento.

In [ ]:
#8. Pruning con Optuna usando API nativa de XGBoost (xgboost.train) + XGBoostPruningCallback
import optuna
from optuna.samplers import TPESampler
from optuna.integration import XGBoostPruningCallback

import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn import set_config
import numpy as np
import joblib

optuna.logging.set_verbosity(optuna.logging.WARNING)
set_config(transform_output="pandas")

# train, val, test
# target = 'quantity'
# numerical_features = ['lat','long','pop','price']  (SIN 'quantity')
# categorical_features = ['city','shop','brand','container','capacity']
features = numerical_features + categorical_features

#  preprocesador por trial (para variar min_frequency)
def _prepro_para_trial(min_freq: float) -> ColumnTransformer:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=min_freq)
    pre = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numerical_features),
            ("cat", ohe, categorical_features),
        ]
    )
    pre.set_output(transform="pandas")
    return pre

class XGBMonotoneBundle:
    def __init__(self, preprocessor, booster):
        self.preprocessor = preprocessor
        self.booster = booster
    def predict(self, X):
        Xtf = self.preprocessor.transform(X[features])
        dm = xgb.DMatrix(Xtf)
        return self.booster.predict(dm)

def objective_pruning(trial: optuna.Trial) -> float:

    learning_rate    = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators     = trial.suggest_int("n_estimators", 50, 1000)
    max_depth        = trial.suggest_int("max_depth", 3, 10)
    max_leaves       = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha        = trial.suggest_float("reg_alpha", 0.0, 1.0)
    reg_lambda       = trial.suggest_float("reg_lambda", 0.0, 1.0)
    ohe_min_freq     = trial.suggest_float("min_frequency", 0.0, 1.0)


    pre_t = _prepro_para_trial(min_freq=ohe_min_freq)
    Xtr = pre_t.fit_transform(X_train)
    ytr = y_train.to_numpy()
    Xva = pre_t.transform(X_val)
    yva = y_val.to_numpy()

    # Restricción monótona: -1 para 'price'
    feat_names = list(Xtr.columns)
    mono_vec = [0] * len(feat_names)
    for i, name in enumerate(feat_names):
        if name == "num__price" or name.endswith("price"):
            mono_vec[i] = -1
    # En xgboost.train, el parámetro debe ir como string "(a,b,c,...)"
    mono_str = "(" + ",".join(str(v) for v in mono_vec) + ")"

    dtrain = xgb.DMatrix(Xtr, label=ytr)
    dvalid = xgb.DMatrix(Xva, label=yva)


    params = {
        "seed": RANDOM_STATE,
        "eta": learning_rate,
        "max_depth": max_depth,
        "max_leaves": max_leaves,
        "min_child_weight": min_child_weight,
        "alpha": reg_alpha,
        "lambda": reg_lambda,
        "tree_method": "hist",
        "eval_metric": "mae",
        "monotone_constraints": mono_str,
    }

    # Pruning callback: monitorea 'validation-mae'
    pruning_cb = XGBoostPruningCallback(trial, "validation-mae")

    # Entrenamiento con pruning
    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=n_estimators,
        evals=[(dvalid, "validation")],
        callbacks=[pruning_cb],
    )

    # MAE final en validación
    yhat_val = booster.predict(dvalid)
    mae = mean_absolute_error(yva, yhat_val)

    # Guardar bundle (preprocessor + booster)
    trial.set_user_attr("model_bundle", XGBMonotoneBundle(pre_t, booster))
    return mae

# Estudio con TPESampler + MedianPruner y timeout 5 min
sampler = TPESampler(seed=RANDOM_STATE)
study_prune = optuna.create_study(
    direction="minimize",
    sampler=sampler,
    pruner=optuna.pruners.MedianPruner()
)
study_prune.optimize(objective_pruning, timeout=300, n_jobs=1, show_progress_bar=True)

# Reporte
best_trial = study_prune.best_trial
print(f"Trials realizados: {len(study_prune.trials)}")
print(f"Mejor MAE (validación): {best_trial.value:.4f}")
print("Mejores hiperparámetros (con pruning):")
for k, v in best_trial.params.items():
    print(f"  - {k}: {v}")

# Guardar el mejor bundle
best_bundle = best_trial.user_attrs["model_bundle"]
joblib.dump(best_bundle, "xgb_monotone_optuna_pruned.pkl")


   0%|          | 00:00/05:00

Streaming output truncated to the last 5000 lines.
[361]	validation-mae:3894.29037
[362]	validation-mae:3892.27080
[363]	validation-mae:3892.80021
[364]	validation-mae:3890.86851
[365]	validation-mae:3889.76116
[366]	validation-mae:3886.78915
[367]	validation-mae:3886.90596
[368]	validation-mae:3885.04578
[369]	validation-mae:3884.39602
[370]	validation-mae:3882.97137
[371]	validation-mae:3881.26097
[372]	validation-mae:3879.89847
[373]	validation-mae:3878.39952
[374]	validation-mae:3877.05545
[375]	validation-mae:3877.57007
[376]	validation-mae:3876.74013
[377]	validation-mae:3875.93313
[378]	validation-mae:3874.30175
[379]	validation-mae:3873.29419
[380]	validation-mae:3872.08082
[381]	validation-mae:3870.23866
[382]	validation-mae:3869.99630
[383]	validation-mae:3868.14191
[384]	validation-mae:3858.22295
[385]	validation-mae:3856.40236
[386]	validation-mae:3851.91464
[387]	validation-mae:3847.44116
[388]	validation-mae:3845.40015
[389]	validation-mae:3838.76437
[390]	validation-mae:

['xgb_monotone_optuna_pruned.pkl']

Al usar Optuna con pruning sobre un XGBRegressor, se realizaron 102 trials (sin pruning se realizaron 244 trials) y se alcanzó un MAE de 3345.00 en el conjunto de validación.

Mejores hiperparámetros encontrados:
learning_rate: 0.08864014465733958
n_estimators: 569
max_depth: 10
max_leaves: 56
min_child_weight: 5
reg_alpha: 0.25005713000393626
reg_lambda: 0.04031678067536146
min_frequency (OneHotEncoder): 0.001668128921520541

learning_rate (0.089). Paso de aprendizaje moderado: favorece una convergencia estable; admite un número mayor de árboles sin sobreajustar de golpe.
max_depth (10). Profundidad elevada que otorga capacidad para capturar interacciones y no linealidades complejas. Se equilibra con min_child_weight y la regularización para contener la varianza.
max_leaves (56). Límite de hojas por árbol que ayuda a acotar la complejidad efectiva. En conjunto con max_depth, restringe el crecimiento de estructuras demasiado ramificadas.
min_child_weight (5). Umbral conservador: impide divisiones con poca evidencia estadística, reduciendo sobreajuste y estabilizando el árbol cuando la profundidad es alta.

Aunque el MAE quedó levemente por sobre el de Optuna solo, el resultado es computacionalmente mejor: el pruning corta temprano configuraciones que considera que no serán buenas, por lo que reduce a menos de la mitad la cantidad de intentos, baja tiempos y costo de cómputo, y mantiene un desempeño muy cercano al anterior.

En cuanto a los hiperparámetros, el learning_rate de 0.089 junto con 569 árboles favorece un aprendizaje progresivo y estable; la profundidad 10 y 56 hojas entregan alta capacidad para capturar no linealidades, mientras que un min_child_weight de 5 actúa como umbral conservador que evita splits con poca evidencia y ayuda a controlar la varianza. La regularización (reg_alpha = 0.25 y reg_lambda = 0.04) apaga ramas débiles y suaviza pesos, y el min_frequency del OneHotEncoder en 0.001668 preserva categorías poco frecuentes, lo que puede aportar señal en nichos a cambio de mayor dimensionalidad. En síntesis, con pruning se obtiene un punto de operación más eficiente en tiempo y recursos, y en consecuencia aumenta el MAE, pero aún así se mantiene cercano al "óptimo".



## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gráfico de historial de optimización [1 punto]
2. Gráfico de coordenadas paralelas [1 punto]
3. Gráfico de importancia de hiperparámetros [1 punto]

Comente sus resultados:

4. ¿Desde qué *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas? [1 punto]
6. ¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo? [0.5 puntos]

In [91]:
# Inserte su código acá

import optuna.visualization as vis

# Gráfico 1: Historial de optimización (evolución del MAE por trial)
fig_history = vis.plot_optimization_history(study_prune)
fig_history.update_layout(title="Historial de Optimización (MAE por trial)")
fig_history.show()

# Gráfico 2: Coordenadas paralelas (relación entre hiperparámetros y MAE)
fig_parallel = vis.plot_parallel_coordinate(study_prune)
fig_parallel.update_layout(title="Coordenadas Paralelas de Hiperparámetros")
fig_parallel.show()

# Gráfico 3: Importancia de hiperparámetros
fig_importance = vis.plot_param_importances(study_prune)
fig_importance.update_layout(title="Importancia de Hiperparámetros en la Optimización")
fig_importance.show()


¿Desde qué trial se empiezan a observar mejoras notables en sus resultados?

Las mejoras notables comienzan desde el trial 1, donde el MAE cae con fuerza desde cerca de ocho mil quinientos hasta alrededor de 5000. Luego aparece un segundo descenso claro entre los trials 12 y 15, cuando el error baja por debajo de 4000. Más adelante se observa otra reducción alrededor del trial 65 hacia la zona de tres mil quinientos y el mejor escalón llega cerca del trial 80, donde el MAE se estabiliza cerca de 3300, después de ese punto los avances son menores.

¿Qué tendencias puede observar a partir del gráfico de coordenadas paralelas?

En el gráfico de coordenadas paralelas, las corridas con menor MAE se concentran cuando el parámetro min_frequency del OneHotEncoder es muy bajo (0.0, 0,1), lo que indica que preservar categorías poco frecuentes beneficia al modelo. También se observa que max_depth se mantiene en valores bajos a medios, aproximadamente entre 3 y 6, combinados con un número de árboles medio a alto, entre 400 y 800, y un learning_rate moderado, entre cero coma cero seis y cero coma uno. La cantidad de hojas se sitúa en rangos intermedios, alrededor de 40 a 80, min_child_weight suele quedar en niveles bajos a medios, entre 1 y 3, y las regularizaciones L1 y L2 se ubican en valores bajos a moderados, aportando control sin rigidizar en exceso.

¿Cuáles son los hiperparámetros con mayor importancia para la optimización de su modelo?

De acuerdo con el gráfico de importancia, el hiperparámetro más influyente es min_frequency, con un peso cercano a 0.58, seguido por max_depth, con un peso cercano a cero coma veintiséis. El resto, como max_leaves, n_estimators, reg_alpha, reg_lambda y learning_rate, tiene una contribución baja, alrededor de 0.03, mientras que min_child_weight es prácticamente irrelevante en esta optimización.

## 6. Síntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validación obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ¿qué modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ¿Existen diferencias con respecto a las métricas obtenidas en el conjunto de validación? ¿Porqué puede ocurrir esto? [1 punto]

In [92]:
# Inserte su código acá

import joblib
from sklearn.metrics import mean_absolute_error
import pandas as pd

# Cargar los modelos
modelos = {
    "Baseline (DummyRegressor)": "dummy_regressor_model.pkl",
    "XGBRegressor base": "xgb_regressor_model.pkl",
    "XGB + Constraint Monótono": "xgb_regressor_monotonic.pkl",
    "XGB + Optuna": "xgb_monotone_optuna.pkl",
    "XGB + Optuna + Pruning": "xgb_monotone_optuna_pruned.pkl"
}

# mae validación
mae_scores = []

for nombre, archivo in modelos.items():
    modelo = joblib.load(archivo)
    preds_val = modelo.predict(X_val)
    mae_val = mean_absolute_error(y_val, preds_val)
    mae_scores.append({"Modelo": nombre, "MAE_validación": mae_val})

# df resumen
tabla_resultados = pd.DataFrame(mae_scores).sort_values("MAE_validación")
tabla_resultados.reset_index(drop=True, inplace=True)

tabla_resultados


/tmp/ipython-input-3837809157.py:9: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/tmp/ipython-input-3837809157.py:9: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/tmp/ipython-input-3837809157.py:9: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.

/tmp/ipython-input-3837809157.py:9: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,Modelo,MAE_validación
0,XGB + Optuna,3301.551025
1,XGB + Optuna + Pruning,3345.001221
2,XGB + Constraint Monótono,3562.329590
3,XGBRegressor base,3642.491943
4,Baseline (DummyRegressor),13503.884737


In [93]:
# modelo en test
best_model = joblib.load("xgb_monotone_optuna.pkl")

# predicciones sobre test
y_test_pred = best_model.predict(X_test)

# MAE en test
mae_test = mean_absolute_error(y_test, y_test_pred)
print(f"MAE sobre conjunto de TEST: {mae_test:.2f}")


MAE sobre conjunto de TEST: 3263.13


/tmp/ipython-input-3837809157.py:9: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



El proceso de mejora sucesiva (baseline → XGB base → restricción monótona → Optuna → Optuna+pruning) mostró que la mayor ganancia proviene de la optimización bayesiana con Optuna (que igualmente contempla constraint monótono), que alcanzó el menor MAE en validación (3301). La evaluación del mismo modelo en test arrojó 3263, con una, lo que indica buena capacidad de generalización y robustece la elección de este modelo como candidato final.

Tal como se pregunta, el MAE en test es mayor que en conjunto de validación, pero esto es esperable ya que el modelo fue seleccionado mediante los resultados obtenidos sobre el set de validación, por lo que se ajusta este conjunto.
El conjunto de test no fue usado antes y dado que es otra porción de los datos, pueden existir cambios en la distribución de los datos y por supuesto, incertidumbre propia de los datos que podría ser evaluado mediante un EDA, pero en este laboratorio casi no se exploraron los datos y solo se vió una muestra al inicio.

# Conclusión
Exito!
<p align="center">
  <img src="https://i.pinimg.com/originals/55/3d/42/553d42bea9b10e0662a05aa8726fc7f4.gif">
</p>